# Week 4 – Source to Bronze Ingestion

This notebook loads the approved batch source files into persistent Bronze Delta tables.

**Project sources**
- `users.csv`
- `subscriptions.csv`
- `sessions.parquet`
- `content_catalog.json`

The Bronze layer preserves source business values and adds ingestion/lineage metadata. No cleansing, Silver transformations, Gold logic, Power BI, Auto Loader, or streaming is included.

## 1. Environment setup

> Replace only the values in the configuration cell with the **approved Unity Catalog catalog, schema, and Volume path** from your project. Do not create a new schema.

The Week 4 guide requires the exact catalog/schema/Volume to be verified before ingestion begins. fileciteturn1file0L30-L44

In [ ]:
%sql
-- Change these three values to your approved project location.
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;

In [ ]:
%sql
-- Change this path to the exact approved Unity Catalog Volume.
-- Example: /Volumes/<catalog>/<schema>/<volume>

CREATE OR REPLACE TEMP VIEW bronze_config AS
SELECT
  '/Volumes/<catalog>/<schema>/<volume>' AS volume_path,
  'W04_ZENAIZ_BRONZE_RUN_01' AS ingestion_run_id,
  'zenaiz_bronze_v1.0' AS schema_version;

SELECT * FROM bronze_config;

## 2. Source inventory

The four uploaded batch files are different formats, so each uses the corresponding Spark reader:
CSV, Parquet, and JSON. The Week 4 instructions require one Bronze table per approved batch source and a source/format/table inventory. fileciteturn1file0L39-L59

In [ ]:
%sql
SELECT * FROM VALUES
  ('users.csv',           'CSV',     'bronze_users'),
  ('subscriptions.csv',  'CSV',     'bronze_subscriptions'),
  ('sessions.parquet',    'Parquet', 'bronze_sessions'),
  ('content_catalog.json','JSON',    'bronze_content_catalog')
AS source_inventory(source_file, source_format, bronze_table);

In [ ]:
%sql
-- File-level check before ingestion.
-- The result should contain all four approved files.

SELECT
  element_at(split(path, '/'), -1) AS file_name,
  path
FROM list_files((SELECT volume_path FROM bronze_config))
WHERE element_at(split(path, '/'), -1) IN
      ('users.csv', 'subscriptions.csv', 'sessions.parquet', 'content_catalog.json');

# 3. Source 1 – Users

**Source:** `users.csv`  
**Format:** CSV  
**Bronze table:** `bronze_users`

Flow: identify source → read → inspect → create Bronze-ready view → write persistent Delta → verify → reconcile.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW users_source
USING CSV
OPTIONS (
  path concat((SELECT volume_path FROM bronze_config), '/users.csv'),
  header 'true',
  inferSchema 'true',
  mode 'PERMISSIVE'
);

In [ ]:
%sql
-- Inspect sample rows, schema and source count.
SELECT * FROM users_source LIMIT 10;

In [ ]:
%sql
DESCRIBE users_source;

SELECT COUNT(*) AS source_count
FROM users_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW users_bronze_ready AS
SELECT
  s.*,
  'users.csv' AS _source_file_name,
  concat((SELECT volume_path FROM bronze_config), '/users.csv') AS _source_file_path,
  current_timestamp() AS _ingested_at,
  (SELECT ingestion_run_id FROM bronze_config) AS _ingestion_run_id,
  (SELECT schema_version FROM bronze_config) AS _schema_version,
  sha2(to_json(struct(s.*)), 256) AS _record_hash
FROM users_source s;

In [ ]:
%sql
SELECT *
FROM users_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_users
USING DELTA
AS
SELECT *
FROM users_bronze_ready;

In [ ]:
%sql
SELECT * FROM bronze_users LIMIT 10;

DESCRIBE TABLE bronze_users;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM users_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_users) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM users_source)
       = (SELECT COUNT(*) FROM bronze_users)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;

# 3. Source 2 – Subscriptions

**Source:** `subscriptions.csv`  
**Format:** CSV  
**Bronze table:** `bronze_subscriptions`

Flow: identify source → read → inspect → create Bronze-ready view → write persistent Delta → verify → reconcile.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW subscriptions_source
USING CSV
OPTIONS (
  path concat((SELECT volume_path FROM bronze_config), '/subscriptions.csv'),
  header 'true',
  inferSchema 'true',
  mode 'PERMISSIVE'
);

In [ ]:
%sql
-- Inspect sample rows, schema and source count.
SELECT * FROM subscriptions_source LIMIT 10;

In [ ]:
%sql
DESCRIBE subscriptions_source;

SELECT COUNT(*) AS source_count
FROM subscriptions_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW subscriptions_bronze_ready AS
SELECT
  s.*,
  'subscriptions.csv' AS _source_file_name,
  concat((SELECT volume_path FROM bronze_config), '/subscriptions.csv') AS _source_file_path,
  current_timestamp() AS _ingested_at,
  (SELECT ingestion_run_id FROM bronze_config) AS _ingestion_run_id,
  (SELECT schema_version FROM bronze_config) AS _schema_version,
  sha2(to_json(struct(s.*)), 256) AS _record_hash
FROM subscriptions_source s;

In [ ]:
%sql
SELECT *
FROM subscriptions_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_subscriptions
USING DELTA
AS
SELECT *
FROM subscriptions_bronze_ready;

In [ ]:
%sql
SELECT * FROM bronze_subscriptions LIMIT 10;

DESCRIBE TABLE bronze_subscriptions;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM subscriptions_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_subscriptions) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM subscriptions_source)
       = (SELECT COUNT(*) FROM bronze_subscriptions)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;

# 3. Source 3 – Sessions

**Source:** `sessions.parquet`  
**Format:** Parquet  
**Bronze table:** `bronze_sessions`

Flow: identify source → read → inspect → create Bronze-ready view → write persistent Delta → verify → reconcile.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sessions_source
USING PARQUET
OPTIONS (
  path concat((SELECT volume_path FROM bronze_config), '/sessions.parquet')
);

In [ ]:
%sql
-- Inspect sample rows, schema and source count.
SELECT * FROM sessions_source LIMIT 10;

In [ ]:
%sql
DESCRIBE sessions_source;

SELECT COUNT(*) AS source_count
FROM sessions_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sessions_bronze_ready AS
SELECT
  s.*,
  'sessions.parquet' AS _source_file_name,
  concat((SELECT volume_path FROM bronze_config), '/sessions.parquet') AS _source_file_path,
  current_timestamp() AS _ingested_at,
  (SELECT ingestion_run_id FROM bronze_config) AS _ingestion_run_id,
  (SELECT schema_version FROM bronze_config) AS _schema_version,
  sha2(to_json(struct(s.*)), 256) AS _record_hash
FROM sessions_source s;

In [ ]:
%sql
SELECT *
FROM sessions_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_sessions
USING DELTA
AS
SELECT *
FROM sessions_bronze_ready;

In [ ]:
%sql
SELECT * FROM bronze_sessions LIMIT 10;

DESCRIBE TABLE bronze_sessions;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM sessions_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_sessions) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM sessions_source)
       = (SELECT COUNT(*) FROM bronze_sessions)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;

# 3. Source 4 – Content Catalog

**Source:** `content_catalog.json`  
**Format:** JSON / JSON Lines  
**Bronze table:** `bronze_content_catalog`

Flow: identify source → read → inspect → create Bronze-ready view → write persistent Delta → verify → reconcile.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW content_catalog_source
USING JSON
OPTIONS (
  path concat((SELECT volume_path FROM bronze_config), '/content_catalog.json'),
  multiLine 'false',
  mode 'PERMISSIVE'
);

In [ ]:
%sql
-- Inspect sample rows, schema and source count.
SELECT * FROM content_catalog_source LIMIT 10;

In [ ]:
%sql
DESCRIBE content_catalog_source;

SELECT COUNT(*) AS source_count
FROM content_catalog_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW content_catalog_bronze_ready AS
SELECT
  s.*,
  'content_catalog.json' AS _source_file_name,
  concat((SELECT volume_path FROM bronze_config), '/content_catalog.json') AS _source_file_path,
  current_timestamp() AS _ingested_at,
  (SELECT ingestion_run_id FROM bronze_config) AS _ingestion_run_id,
  (SELECT schema_version FROM bronze_config) AS _schema_version,
  sha2(to_json(struct(s.*)), 256) AS _record_hash
FROM content_catalog_source s;

In [ ]:
%sql
SELECT *
FROM content_catalog_bronze_ready
LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_content_catalog
USING DELTA
AS
SELECT *
FROM content_catalog_bronze_ready;

In [ ]:
%sql
SELECT * FROM bronze_content_catalog LIMIT 10;

DESCRIBE TABLE bronze_content_catalog;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM content_catalog_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_content_catalog) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM content_catalog_source)
       = (SELECT COUNT(*) FROM bronze_content_catalog)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;

# 7. Consolidated reconciliation

The Week 4 guide requires one consolidated source-versus-Bronze reconciliation for all approved batch sources. fileciteturn1file1L79-L84

In [ ]:
%sql
WITH counts AS (
  SELECT 'users' AS dataset,
         (SELECT COUNT(*) FROM users_source) AS source_count,
         (SELECT COUNT(*) FROM bronze_users) AS bronze_count
  UNION ALL
  SELECT 'subscriptions',
         (SELECT COUNT(*) FROM subscriptions_source),
         (SELECT COUNT(*) FROM bronze_subscriptions)
  UNION ALL
  SELECT 'sessions',
         (SELECT COUNT(*) FROM sessions_source),
         (SELECT COUNT(*) FROM bronze_sessions)
  UNION ALL
  SELECT 'content_catalog',
         (SELECT COUNT(*) FROM content_catalog_source),
         (SELECT COUNT(*) FROM bronze_content_catalog)
)
SELECT
  dataset,
  source_count,
  bronze_count,
  bronze_count - source_count AS count_difference,
  CASE WHEN source_count = bronze_count THEN 'MATCH'
       ELSE 'CHECK' END AS status
FROM counts
ORDER BY dataset;

# 8. Bronze metadata validation

Every Bronze table should contain the source filename, ingestion timestamp, controlled run ID, schema version and record hash. fileciteturn1file0L53-L59

In [ ]:
%sql
SELECT 'bronze_users' AS table_name,
       COUNT(*) AS rows,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END) AS missing_path,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END) AS missing_schema_version,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_users
UNION ALL
SELECT 'bronze_subscriptions', COUNT(*),
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_subscriptions
UNION ALL
SELECT 'bronze_sessions', COUNT(*),
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_sessions
UNION ALL
SELECT 'bronze_content_catalog', COUNT(*),
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_content_catalog;

# 9. Safe rerun proof

This load uses `CREATE OR REPLACE TABLE AS SELECT`. Rerunning the complete source section replaces the current Bronze snapshot rather than appending the same batch again, so the Bronze count should remain equal to the source count.

Run the selected source section again, then execute the following check. The Week 4 guide specifically requires proof that a rerun does not unintentionally double the Bronze count. fileciteturn1file1L79-L84

In [ ]:
%sql
-- Rerun proof for users.
-- Re-execute the "Source 1 – Users" section above first.

SELECT
  (SELECT COUNT(*) FROM users_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_users) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM users_source)
       = (SELECT COUNT(*) FROM bronze_users)
    THEN 'SAFE_RERUN_MATCH'
    ELSE 'CHECK'
  END AS rerun_status;

In [ ]:
%sql
DESCRIBE HISTORY bronze_users;

# 10. Confirm Bronze tables

Run this after all four sections succeed.

In [ ]:
%sql
SHOW TABLES;

SELECT table_name
FROM information_schema.tables
WHERE table_schema = current_schema()
  AND table_name IN (
    'bronze_users',
    'bronze_subscriptions',
    'bronze_sessions',
    'bronze_content_catalog'
  )
ORDER BY table_name;

## Week 4 completion checklist

- [ ] Approved files are visible in the Unity Catalog Volume.
- [ ] Correct reader is used for each format.
- [ ] Source business values are preserved.
- [ ] Four persistent Bronze Delta tables exist.
- [ ] Required ingestion/lineage metadata is present.
- [ ] Source and Bronze counts match.
- [ ] One complete source section was rerun safely.
- [ ] Delta history was captured.
- [ ] No Silver, Gold, streaming, Auto Loader or cleansing work was added.

The Week 4 boundary ends after verified Bronze ingestion, reconciliation, rerun proof and evidence. fileciteturn1file1L101-L119